In [23]:
from models import SEASModel
from evaluation_utils import evaluate_iplg_convergence
import GridMLM_tokenizers
from GridMLM_tokenizers import CSGridMLMTokenizer
from data_utils import CSGridMLMDataset
from train_utils import full_to_partial_masking
from torch.utils.data import DataLoader
import torch
from torch.nn import CrossEntropyLoss
import os
import pickle
from train_utils import train_IPLG
from data_utils import latent_MH_collate_fn
from pprint import pprint
import pandas as pd

device_name = 'cuda:0'
batch_size = 4

In [24]:
if device_name == 'cpu':
    device = torch.device('cpu')
else:
    if torch.cuda.is_available():
        device = torch.device(device_name)
    else:
        print('Selected device not available: ' + device_name)
# end device selection

In [25]:
tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)
mask_token_id = tokenizer.mask_token_id
bar_token_id = tokenizer.bar_token_id

In [26]:
loss_scheme = 'l'

d_model = 512
transformer_model = SEASModel(
    chord_vocab_size=len(tokenizer.vocab),
    d_model=d_model,
    nhead=8,
    num_layers=8,
    grid_length=80,
    pianoroll_dim=tokenizer.pianoroll_dim,
    guidance_dim=d_model,
    device=device,
)
checkpoint = torch.load(f'saved_models/iplg/SE/iplg_{loss_scheme}_loss.pt', map_location=device_name)
transformer_model.load_state_dict(checkpoint)
transformer_model.to(device)
transformer_model.eval()

SEASModel(
  (melody_proj): Linear(in_features=13, out_features=512, bias=True)
  (harmony_embedding): Embedding(355, 512)
  (dropout): Dropout(p=0.3, inplace=False)
  (encoder_layers): ModuleList(
    (0-7): 8 x TransformerEncoderLayerWithFiLM(
      (layer): TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=2048, bias=True)
        (dropout): Dropout(p=0.3, inplace=False)
        (linear2): Linear(in_features=2048, out_features=512, bias=True)
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.3, inplace=False)
        (dropout2): Dropout(p=0.3, inplace=False)
      )
      (film): FiLMAdapter(
        (gamma): Linear(in_features=512, out_features=512, bias=True)
        (beta): Linea

In [27]:
source_folder = 'MIDIs/jazz/real'
target_folder = 'MIDIs/nottingham/real'

source_piece_path = os.path.join(source_folder, 'real_0.mid')
target_piece_path = os.path.join(target_folder, 'real_0.mid')

In [28]:
source_encoded = tokenizer.encode(source_piece_path)
target_encoded = tokenizer.encode(target_piece_path)

In [29]:
# get the source info
source_melody_grid = torch.tensor(source_encoded['pianoroll']).unsqueeze(0).to(device)
source_harmony_ids = torch.tensor(source_encoded['harmony_ids']).unsqueeze(0).to(device)

In [30]:
# for source, we need to consider harmony fully masked, except from bar tokens
source_harmony_masks, _ = full_to_partial_masking(
                    source_harmony_ids,
                    mask_token_id,
                    0,
                    bar_token_id=bar_token_id
                )

In [31]:
# get the "natural" harmony embeddings for this melody - consider mask harmony tokens
_, source_melody_layers_output = transformer_model(source_melody_grid, source_harmony_masks, get_layers_output=True)

In [32]:
print(source_melody_layers_output[0].shape)

torch.Size([1, 1, 512])


In [33]:
target_melody_grid = torch.tensor(target_encoded['pianoroll']).unsqueeze(0).to(device)
target_harmony_ids = torch.tensor(target_encoded['harmony_ids']).unsqueeze(0).to(device)

In [34]:
# for the target harmony, we need both melody and harmony token ids inside
_, target_melody_harmony_layers_output = transformer_model(target_melody_grid, target_harmony_ids, get_layers_output=True)

In [35]:
# get the difference as activation steering
h = {}
for k,v in target_melody_harmony_layers_output.items():
    h[k] = v - source_melody_layers_output[k]

In [36]:
h_4 = {4: h[4]}
h_6 = {6: h[6]}

In [37]:
# generate 
y = transformer_model(source_melody_grid, source_harmony_masks, steering_vectors=h_4, alpha=0.5, get_layers_output=False)

In [38]:
# get average steering from folder
target_files = os.listdir(target_folder)

target_mean_h = {}

for f in target_files:
    target_piece_path = os.path.join( target_folder, f )
    target_encoded = tokenizer.encode(target_piece_path)
    _, target_melody_harmony_layers_output = transformer_model(target_melody_grid, target_harmony_ids, get_layers_output=True)
    for k,v in target_melody_harmony_layers_output.items():
        if k in target_mean_h.keys():
            target_mean_h[k] += v/len(target_files)
        else:
            target_mean_h[k] = v/len(target_files)

In [39]:
from evaluation_utils import evaluate_iplg_convergence
import GridMLM_tokenizers
from GridMLM_tokenizers import CSGridMLMTokenizer
from data_utils import CSGridMLMDataset
from models import SEFiLMModel
import torch
from torch.nn import CrossEntropyLoss
import os
import pickle
from train_utils import train_IPLG
from pprint import pprint
import pandas as pd

device_name = 'cuda:0'
batch_size = 4

In [40]:
data_files = ('nottingham_test.pickle', 'gjt_CA_test.pickle')

home_path = f'data/latent_datasets/SE/{data_files[0]}'
foreign_path = f'data/latent_datasets/SE/{data_files[1]}'

In [41]:
with open(home_path, 'rb') as f:
    home_dataset = pickle.load(f)
with open(foreign_path, 'rb') as f:
    foreign_dataset = pickle.load(f)

if device_name == 'cpu':
    device = torch.device('cpu')
else:
    if torch.cuda.is_available():
        device = torch.device(device_name)
    else:
        print('Selected device not available: ' + device_name)
# end device selection

latent_loss_fn = torch.nn.MSELoss()

tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)
mask_token_id = tokenizer.mask_token_id
bar_token_id = tokenizer.bar_token_id

logits_loss_fn =CrossEntropyLoss(ignore_index=-100)

In [42]:
home_loader = DataLoader(
    home_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
    collate_fn=latent_MH_collate_fn
)
foreign_loader = DataLoader(
    foreign_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
    collate_fn=latent_MH_collate_fn
)

In [43]:
batch = next(iter(home_loader))
foreign_batch = next(iter(foreign_loader))

In [44]:
from evaluation_utils import make_mixed_batch

In [ ]:
def get_actisteer_guidance(
        model,
        bar_token_id,
        mask_token_id,
        source_melody,
        source_harmony,
        target_melody,
        target_harmony
    ):
    # for source, we need to consider harmony fully masked, except from bar tokens
    source_harmony_masks, _ = full_to_partial_masking(
        source_harmony,
        mask_token_id,
        0,
        bar_token_id=bar_token_id
    )
    # get the "natural" harmony embeddings for this melody - consider mask harmony tokens
    _, source_melody_layers_output = model(
        source_melody,
        source_harmony_masks, 
        get_layers_output=True
    )
    # for the target harmony, we need both melody and harmony token ids inside
    _, target_melody_harmony_layers_output = model(
        target_melody,
        target_harmony,
        get_layers_output=True
    )
    # get the difference as activation steering
    h = {}
    for k,v in target_melody_harmony_layers_output.items():
        h[k] = v - source_melody_layers_output[k]
        h[k] = h[k] / (h[k].norm() + 1e-6)
    return h
# end get_actisteer_guidance

In [56]:
melody_grid = batch["pianoroll"].to(device)
harmony_gt = batch["harmony_ids"].to(device)
# home_guidance_embeddings = batch["latent"].to(device)
home_guidance_dict = get_actisteer_guidance(
    transformer_model,
    bar_token_id,
    melody_grid,
    harmony_gt,
    melody_grid,
    harmony_gt
)
foreign_batch = next(iter(foreign_loader))
mixed_batch_1 = make_mixed_batch(foreign_batch, "latent")
mixed_batch_2 = make_mixed_batch(foreign_batch, "latent")

foreign_guidance_dict = get_actisteer_guidance(
    transformer_model,
    bar_token_id,
    melody_grid,
    harmony_gt,
    mixed_batch_1["pianoroll"].to(device),
    mixed_batch_1["harmony_ids"].to(device)
)

In [57]:
print(home_guidance_dict)

{0: tensor([[[-0.1605,  0.0671,  0.1482,  ...,  0.3417, -0.3613,  0.7631]],

        [[-0.2858,  0.1605,  0.0163,  ...,  0.2959, -0.2267,  0.5910]],

        [[-0.1225,  0.0918,  0.1390,  ...,  0.2194, -0.4274,  0.3923]],

        [[-0.1622,  0.2238,  0.3132,  ...,  0.2746, -0.2796,  0.7755]]],
       device='cuda:0', grad_fn=<SubBackward0>), 1: tensor([[[-0.1465,  0.0488,  0.0391,  ...,  0.0175, -0.3212,  0.2769]],

        [[-0.1077,  0.0907, -0.0919,  ...,  0.0715, -0.2284,  0.1945]],

        [[-0.0329,  0.1074, -0.2134,  ...,  0.1360, -0.3956,  0.2811]],

        [[-0.1217,  0.2227,  0.0797,  ...,  0.0211, -0.4480,  0.4816]]],
       device='cuda:0', grad_fn=<SubBackward0>), 2: tensor([[[-0.1767, -0.1732,  0.1322,  ...,  0.1667, -0.1576, -0.0715]],

        [[-0.1153, -0.0713,  0.0117,  ...,  0.2675, -0.3167, -0.2859]],

        [[-0.1622, -0.1317,  0.0371,  ...,  0.0706, -0.2954, -0.0533]],

        [[-0.0727, -0.0325,  0.0637,  ...,  0.0637, -0.4402,  0.0087]]],
       device='c